In [ ]:
# =============================================================================
# REINFORCEMENT LEARNING FOUNDATIONS — learn by doing, not by labels
# =============================================================================
#
# So far in this course:
#   Supervised learning  →  (input, correct answer) pairs. "Here's a 7, copy it."
#   Generative models    →  make new data. No single right pixel per step.
#
# Reinforcement learning (RL) is different:
#   An AGENT lives in a WORLD, picks ACTIONS, gets REWARDS (or penalties).
#   Nobody hands you the perfect move at each step. You discover it by trial.
#
# Think: teaching a dog tricks.
#   State   = where the dog is, what it's doing (sitting? standing?)
#   Action  = you say "sit" or "stay" or give a treat timing
#   Reward  = treat (+1) or nothing (0) or scold (-1)
#   Policy  = the dog's habit: "when I hear sit, I lower my hips"
#
# Or a video game:
#   State   = screen / position / health
#   Action  = jump, shoot, move left
#   Reward  = +10 for coin, -100 for dying
#   Goal    = maximize TOTAL reward over the whole episode, not one frame
#
# The hard part: actions have DELAYED consequences. Jump now → land on platform
# 2 seconds later → reach the flag → big reward. RL must connect those dots.
#
#
# ---------------------------------------------------------------------------
# 1) MARKOV DECISION PROCESS (MDP) — the board game's rulebook
# ---------------------------------------------------------------------------
#
# An MDP is the standard math picture of an RL problem. Five ingredients:
#
#   S  = States      (where can you be?)
#   A  = Actions     (what can you do in each state?)
#   P  = Dynamics    (if I'm in s, do a, where do I land next?)
#   R  = Rewards     (how good was that step?)
#   γ  = Gamma       (discount: how much do I care about FAR-future reward?)
#
# Tiny grid example (4 rooms):
#
#     [Start] --right--> [ ] --right--> [Goal +10]
#        |                  |
#      down               down
#        v                  v
#       [ ]               [Pit -10]
#
#   S = {start, middle, goal, pit, ...}
#   A = {up, down, left, right}  (maybe illegal into walls)
#   P(s' | s, a) = probability of landing in s' after action a in s
#   R(s, a, s')  = reward for that transition (goal +10, pit -10, else 0)
#
# THE MARKOV PROPERTY (the name "Markov" is doing real work):
#   "The future depends on the past ONLY through the present state."
#
#   In English: the current state s_t should already contain everything
#   that matters for deciding what happens next. You don't need the full
#   history of every move since kindergarten.
#
#   Good state: chess board position (all pieces visible).
#   Bad state:  "I'm on square e4" without knowing where the rest of the
#               pieces are — you've hidden information the game needs.
#
#   If your state is Markov, then:
#     P(s_{t+1} | s_t, a_t, s_{t-1}, a_{t-1}, ...)  =  P(s_{t+1} | s_t, a_t)
#
# EPISODE vs CONTINUING:
#   Episodic  = game ends (win/lose). Reset to start. Mario level, chess game.
#   Continuing = runs forever (or very long). Factory control, stock trading.
#
# POLICY π (pi):
#   π(a | s) = probability of choosing action a in state s
#   Deterministic policy: one fixed action per state ("always go right at start")
#   Stochastic policy: sometimes explore ("usually right, sometimes try down")
#
# RETURN G_t (total future reward from time t):
#   G_t = R_{t+1} + γ R_{t+2} + γ² R_{t+3} + ...
#
#   γ ∈ [0, 1] is the DISCOUNT:
#     γ = 0   → only care about immediate next reward (myopic)
#     γ = 1   → future counts fully (only ok if episodes are guaranteed short)
#     γ ≈ 0.99 → standard in long games; far future still matters, but less
#
#   Why discount? (1) math converges, (2) humans prefer bird-in-hand,
#   (3) in infinite horizons, undiscounted sums can blow up to infinity.
#
# VALUE FUNCTIONS — "how good is it to be here?"
#
#   V^π(s) = expected return if you start in s and follow policy π forever
#            "If I'm in this state and keep acting like π, how much reward
#             do I collect on average?"
#
#   Q^π(s, a) = expected return if you start in s, take action a FIRST,
#               then follow π
#            "If I force this one move a, then behave normally, what's the
#             average score?"
#
#   Q is often more useful for control: pick argmax_a Q(s,a) = best move now.
#   V tells you how good a STATE is; Q tells you how good a STATE–ACTION pair is.
#
#
# ---------------------------------------------------------------------------
# 2) BELLMAN EQUATIONS — recursive "value of here = reward + value of next"
# ---------------------------------------------------------------------------
#
# The big insight: optimal future value can be written in terms of ITSELF.
# Like compound interest, but for states.
#
# BELLMAN EXPECTATION (for a fixed policy π):
#
#   V^π(s) = Σ_a π(a|s) Σ_{s'} P(s'|s,a) [ R(s,a,s') + γ V^π(s') ]
#
# English:
#   "Value of being in s under π =
#      average over actions π might take
#      of (immediate reward + discounted value of wherever you land)"
#
# Same idea for Q:
#
#   Q^π(s,a) = Σ_{s'} P(s'|s,a) [ R(s,a,s') + γ Σ_{a'} π(a'|s') Q^π(s',a') ]
#
# BELLMAN OPTIMALITY (for the BEST possible behavior, π*):
#
#   V*(s) = max_a Σ_{s'} P(s'|s,a) [ R(s,a,s') + γ V*(s') ]
#
#   Q*(s,a) = Σ_{s'} P(s'|s,a) [ R(s,a,s') + γ max_{a'} Q*(s',a') ]
#
# English:
#   "Best value here = best action's (reward now + discounted best value next)"
#
# The max in optimal Q is why RL is CONTROL: at each state, pick the action
# with highest Q*(s,a). No teacher needed at runtime — the table is the teacher.
#
# WHY THIS MATTERS:
#   Bellman turns "infinite future" into a LOCAL equation you can solve or
#   learn iteratively. Every algorithm in this phase is a cousin of:
#     "current estimate ← reward + γ × next estimate"
#
#
# ---------------------------------------------------------------------------
# 3) DYNAMIC PROGRAMMING (DP) — solve the MDP when you KNOW the rules
# ---------------------------------------------------------------------------
#
# DP = planning with perfect knowledge of P and R. You don't learn from
# messy real-world interaction; you crunch the math on paper (or in code).
#
# Assumption: full model of the environment (transition probabilities).
# Tiny grids, toy MDPs, known games — yes. Atari from pixels — no (later).
#
# Two classic algorithms (both use Bellman backups):
#
# (A) POLICY ITERATION
#     1. Start with any policy π (even random).
#     2. POLICY EVALUATION: compute V^π for that π (solve Bellman expectation).
#     3. POLICY IMPROVEMENT: greedily set π(s) = argmax_a Q(s,a) from that V.
#     4. Repeat until π stops changing.
#
#     Like: draft a game plan → score how good it is → fix obvious mistakes
#           → repeat until the plan is stable.
#
# (B) VALUE ITERATION
#     Skip keeping π explicit. Repeatedly apply the Bellman OPTIMALITY update:
#
#       V_{k+1}(s) ← max_a Σ_{s'} P(s'|s,a) [ R + γ V_k(s') ]
#
#     Until V stops changing. Then π*(s) = argmax_a [...] from final V.
#
#     Often fewer concepts; one loop that directly chases V*.
#
# DP limitations (why we don't stop here):
#   • Need P(s'|s,a) — real robots / games rarely give you the full table.
#   • Curse of dimensionality: |S| or |A| huge → table doesn't fit in memory.
#   • DP is the FOUNDATION; MC and TD (later) relax "known model" and scale up.
#
#
# ---------------------------------------------------------------------------
# 4) MONTE CARLO (MC) METHODS — learn from finished episodes, no model needed
# ---------------------------------------------------------------------------
#
# "Monte Carlo" = estimate expectations by AVERAGING many random rollouts.
# No Bellman model of P required. You only need to PLAY the game.
#
# Core idea:
#   1. Follow policy π, run a FULL episode until done: s0,a0,r1,s1,a1,r2,...
#   2. Compute actual return G_t for each visited step t.
#   3. Update estimates:
#        V(s) ← average of all G_t you ever saw starting from s
#        Q(s,a) ← average of all G_t you ever saw after taking a in s
#
# First-visit MC: only count G_t the FIRST time s appears in an episode.
# Every-visit MC: count every time s appears (still works with enough data).
#
# Example intuition:
#   State = "blackjack hand 16 vs dealer 10"
#   Action = hit or stand
#   Run 10,000 full hands with a sticky policy, record profit each time.
#   Q(16, hit)  ≈ average profit when you hit from 16
#   Q(16, stand) ≈ average profit when you stand
#   Pick the higher average → better policy emerges without knowing card math.
#
# MC strengths:
#   • No model of environment dynamics needed.
#   • Unbiased estimates of V^π / Q^π (given enough episodes).
#   • Simple to explain: "try many times, average what happened."
#
# MC weaknesses:
#   • Must wait until episode ENDS to get G_t (bad for very long episodes).
#   • High variance: one lucky episode skews averages early on.
#   • Exploring states you rarely visit is slow (need many episodes).
#
# EXPLORATION (brief — every RL chapter needs this):
#   If you only do what looks best so far, you never discover a better move.
#   ε-greedy: with prob ε pick random action; else pick best Q(s,a).
#   ε starts high (explore), often decays over time (exploit what you learned).
#
#
# ---------------------------------------------------------------------------
# How the four pieces fit together
# ---------------------------------------------------------------------------
#
#   MDP          → problem definition (states, actions, rewards, γ)
#   Bellman      → recursive truth relating value today vs value tomorrow
#   DP           → exact solution when P and R are known (small tabular worlds)
#   Monte Carlo  → learn values from sample episodes when P is unknown
#
#   Later in the course you'll see:
#     TD learning  → update after EVERY step (bootstrap; don't wait for end)
#     Q-learning   → off-policy control (learn optimal while exploring)
#     Deep Q / PPO → same ideas, but V and Q are neural nets, not tables
#
#
# ---------------------------------------------------------------------------
# One table
# ---------------------------------------------------------------------------
#   Object        What it is                          Symbol
#   ------------  ----------------------------------  -----------------
#   State         "where am I?"                       s ∈ S
#   Action        "what can I do?"                    a ∈ A
#   Reward        immediate score                     r or R(s,a,s')
#   Policy        behavior rule                       π(a|s)
#   Return        discounted sum of future rewards    G_t
#   Value         expected return from a state        V^π(s)
#   Action-value  expected return from (s,a)        Q^π(s,a)
#   Discount      weight on future rewards            γ
#   Markov        future ⊥ past given present s       P(s'|s,a)
#
#   Method        Needs model P?   Updates when?
#   ------------  ---------------  ---------------------------------
#   DP            YES              sweep all states (planning)
#   Monte Carlo   NO               end of episode (full return G_t)
#   TD (later)    NO               each step (bootstrap next value)
#
# Next cells: small grid MDP, Bellman backups in code, DP solvers, MC estimates.
#


In [ ]:
# =============================================================================
# GRID WORLD — a tiny MDP we can solve on paper AND simulate
# =============================================================================
#
# This class is TWO things at once:
#
#   (1) MODEL for Dynamic Programming
#       self.P stores every transition P(s'|s,a), reward, and done flag.
#       DP can sweep Bellman updates without ever "playing" the game.
#
#   (2) ENVIRONMENT for Monte Carlo
#       reset() + step(action) let an agent walk around and collect episodes.
#       MC never looks at self.P — it only sees (state, action, reward) rolls.
#
# Grid layout (size=4, 0-indexed rows/cols):
#
#     (0,0) (0,1) (0,2) (0,3)
#     (1,0)  ##   (1,2) (1,3)     ## = obstacle at (1,1)
#     (2,0) (2,1)  ##   (2,3)     ## = obstacle at (2,2)
#     (3,0) (3,1) (3,2) [GOAL]    goal = terminal at (3,3)
#
# Reward design: -1 every step, 0 at goal. Agent wants SHORT paths (fewer -1s).
# γ = 0.9 discounts how much we care about rewards many steps away.
#

import numpy as np
import matplotlib.pyplot as plt
import random


class GridWorld:
    def __init__(self, size=4):
        # --- S: state space ---------------------------------------------------
        self.size = size
        # State = (row, col). Tuple is hashable → can be dict keys for V, Q, π.
        self.states = [(i, j) for i in range(size) for j in range(size)]

        # Terminal = episode ends here. No more actions, V(goal)=0 by convention.
        self.terminal = (size - 1, size - 1)  # bottom-right corner

        # Obstacles = never enter, never start here. Skipped in P and reset().
        self.obstacles = [(1, 1), (2, 2)]

        # --- A: action space --------------------------------------------------
        self.actions = ["up", "down", "left", "right"]
        # How each action moves (Δrow, Δcol). "up" decreases row (toward top).
        self.action_effects = {
            "up": (-1, 0),
            "down": (1, 0),
            "left": (0, -1),
            "right": (0, 1),
        }

        # --- R and γ from the intro cell --------------------------------------
        self.gamma = 0.9
        self.reward_step = -1.0      # living cost: every move hurts a little
        self.reward_terminal = 0.0  # stepping ONTO goal (not used in step yet)

        # --- P: full transition model for DP ----------------------------------
        # Format: P[s][a] = list of (prob, next_state, reward, done)
        # This grid is DETERMINISTIC: prob is always 1.0 for one outcome.
        # Stochastic envs would list multiple (prob, s', r, done) tuples.
        self.P = {}
        for s in self.states:
            if s == self.terminal or s in self.obstacles:
                continue  # no actions from terminal or inside walls
            self.P[s] = {}
            for a in self.actions:
                next_s = self._get_next_state(s, a)
                reward = self.reward_step
                done = next_s == self.terminal
                self.P[s][a] = [(1.0, next_s, reward, done)]

    def _get_next_state(self, state, action):
        """Physics: one move. Bounce in place if you hit wall or obstacle."""
        if state == self.terminal or state in self.obstacles:
            return state
        i, j = state
        di, dj = self.action_effects[action]
        ni, nj = i + di, j + dj
        in_bounds = 0 <= ni < self.size and 0 <= nj < self.size
        if in_bounds and (ni, nj) not in self.obstacles:
            return (ni, nj)
        return state  # illegal move → stay (still pay -1 reward in step())

    def reset(self):
        """
        Start a new episode (for MC rollouts).
        Random non-terminal, non-obstacle cell — so MC sees many start states.
        """
        available = [
            s for s in self.states if s != self.terminal and s not in self.obstacles
        ]
        self.current_state = random.choice(available)
        return self.current_state

    def step(self, action):
        """
        Gym-style API: take action, return (next_state, reward, done, info).
        MC loops: s = reset(); while not done: s,r,done,_ = step(a)
        """
        assert self.current_state != self.terminal, "Episode already finished"
        next_s = self._get_next_state(self.current_state, action)
        reward = self.reward_step
        done = next_s == self.terminal
        self.current_state = next_s
        return next_s, reward, done, {}


# --- quick sanity check + picture --------------------------------------------
env = GridWorld(size=4)
print(f"States: {len(env.states)}  |  Actions: {env.actions}")
print(f"Terminal: {env.terminal}  |  Obstacles: {env.obstacles}")
print(f"Non-terminal states in P (for DP): {len(env.P)}")
print("Example transition P[(0,0)]['right']:", env.P[(0, 0)]["right"])

# One random episode (3 steps) to show reset/step
s = env.reset()
print(f"\nMC demo — start at {s}")
for _ in range(3):
    a = random.choice(env.actions)
    s, r, done, _ = env.step(a)
    print(f"  action={a:5s} → state={s}  reward={r}  done={done}")
    if done:
        break

# Draw the board
grid = np.zeros((env.size, env.size))
for (i, j) in env.obstacles:
    grid[i, j] = -1  # wall
grid[env.terminal] = 2  # goal
plt.imshow(grid, cmap="RdYlGn", vmin=-1, vmax=2)
for i in range(env.size):
    for j in range(env.size):
        if (i, j) in env.obstacles:
            plt.text(j, i, "X", ha="center", va="center", fontsize=14, color="white")
        elif (i, j) == env.terminal:
            plt.text(j, i, "G", ha="center", va="center", fontsize=14, fontweight="bold")
        else:
            plt.text(j, i, f"{i},{j}", ha="center", va="center", fontsize=7, color="gray")
plt.title("GridWorld — green=goal, red=obstacle, labels=state (row,col)")
plt.axis("off")
plt.show()

# ---------------------------------------------------------------------------
# HOW TO READ
# ---------------------------------------------------------------------------
# P table: DP's cheat sheet. policy_evaluation() reads env.P[s][a] and never
# calls step(). That's "planning with a known model."
#
# reset/step: MC's way to generate data. Agent doesn't need P — it learns from
# actual returns G_t after walking to the goal.
#
# -1/step: optimal policy should reach G in as few steps as possible, winding
# around obstacles instead of bouncing off walls forever.
#

In [ ]:
# =============================================================================
# DYNAMIC PROGRAMMING — solve the MDP when we KNOW the rules (env.P)
# =============================================================================
#
# Recap from the intro:
#   Bellman expectation:  V^π(s) = Σ  P(s'|s,a) [ R + γ V^π(s') ]   (a from π)
#   Bellman optimality:   V*(s)  = max_a Σ P(s'|s,a) [ R + γ V*(s') ]
#
# DP = repeatedly apply those equations using the full model env.P.
# We never call reset()/step() here. That is PLANNING, not learning from play.
#
# Two algorithms:
#   Policy iteration  = evaluate π → improve π → repeat until π stable
#   Value iteration   = only update V with the max_a Bellman backup, then
#                       read off π at the end
#
# Both need env.P (built in GridWorld). Monte Carlo (next) will NOT.
#


def policy_evaluation(env, policy, theta=1e-6):
    """
    Score a FIXED policy π: keep sweeping Bellman EXPECTATION until V settles.

    English loop:
      for every state s:
        look up the ONE action π says to take
        V(s) ← reward + γ × V(next)   (averaged if transitions are random)

    theta: stop when no state changes by more than this (convergence check).
    """
    # Obstacles have no value; terminal stays 0 (episode over → no more return).
    V = {s: 0.0 for s in env.states if s not in env.obstacles}
    while True:
        delta = 0.0  # biggest |change| this sweep
        for s in V:
            if s == env.terminal:
                V[s] = 0.0
                continue
            v_old = V[s]
            a = policy[s]  # FIXED action from π — no max here
            transitions = env.P[s][a]
            # Bellman expectation backup (deterministic → one term with prob=1)
            v_new = 0.0
            for prob, next_s, reward, done in transitions:
                v_new += prob * (reward + env.gamma * V[next_s])
            V[s] = v_new
            delta = max(delta, abs(v_old - V[s]))
        if delta < theta:
            break  # V^π found for this policy
    return V


def policy_improvement(env, V):
    """
    Greedy one-step look-ahead: for each s, try EVERY action, keep the best.

      Q(s,a) = Σ P(s'|s,a) [ R + γ V(s') ]
      π(s)   = argmax_a Q(s,a)

    If V was already V^π, this makes a NEW policy that is at least as good
    (policy improvement theorem). Often strictly better until π*.
    """
    policy = {}
    for s in env.P:  # only non-terminal, non-obstacle states
        q_values = {}
        for a in env.actions:
            q = 0.0
            for prob, next_s, reward, done in env.P[s][a]:
                q += prob * (reward + env.gamma * V[next_s])
            q_values[a] = q
        policy[s] = max(q_values, key=q_values.get)  # greedy action
    return policy


def policy_iteration(env):
    """
    Alternating evaluate ↔ improve until the policy stops changing.

      π_0 (random) → V^{π_0} → π_1 → V^{π_1} → … → π* , V*

    Like: draft a game plan → score it → fix bad moves → repeat.
    """
    policy = {s: random.choice(env.actions) for s in env.P}
    while True:
        V = policy_evaluation(env, policy)
        new_policy = policy_improvement(env, V)
        if new_policy == policy:
            break  # greedy w.r.t. V^π is still π → optimal
        policy = new_policy
    return policy, V


def value_iteration(env, theta=1e-6):
    """
    Skip keeping π during the loop. Directly chase V* with Bellman OPTIMALITY:

      V(s) ← max_a  Σ P(s'|s,a) [ R + γ V(s') ]

    When V settles, derive π*(s) = argmax_a Q(s,a) once at the end.
    Same destination as policy iteration; usually fewer conceptual pieces.
    """
    V = {s: 0.0 for s in env.states if s not in env.obstacles}
    while True:
        delta = 0.0
        for s in V:
            if s == env.terminal:
                V[s] = 0.0
                continue
            v_old = V[s]
            best_value = -float("inf")
            for a in env.actions:  # try ALL actions → max_a (optimality)
                q = 0.0
                for prob, next_s, reward, done in env.P[s][a]:
                    q += prob * (reward + env.gamma * V[next_s])
                best_value = max(best_value, q)
            V[s] = best_value
            delta = max(delta, abs(v_old - V[s]))
        if delta < theta:
            break

    # Extract greedy policy from the final V*
    policy = {}
    for s in env.P:
        q_values = {}
        for a in env.actions:
            q = 0.0
            for prob, next_s, reward, done in env.P[s][a]:
                q += prob * (reward + env.gamma * V[next_s])
            q_values[a] = q
        policy[s] = max(q_values, key=q_values.get)
    return policy, V


# --- run both solvers on the GridWorld from the previous cell ----------------
try:
    env
except NameError:
    raise RuntimeError("Run the GridWorld cell first so `env` exists.")

pi_pi, V_pi = policy_iteration(env)
pi_vi, V_vi = value_iteration(env)

print("Policy iteration  π*:")
for s in sorted(pi_pi):
    print(f"  {s} → {pi_pi[s]:5s}   V≈{V_pi[s]:.2f}")
print("\nValue iteration π* (should match / be equivalent):")
for s in sorted(pi_vi):
    print(f"  {s} → {pi_vi[s]:5s}   V≈{V_vi[s]:.2f}")

# Heatmap of V* + arrows for the policy
arrow = {"up": "↑", "down": "↓", "left": "←", "right": "→"}
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, title, policy, V in [
    (axes[0], "Policy iteration V*", pi_pi, V_pi),
    (axes[1], "Value iteration V*", pi_vi, V_vi),
]:
    board = np.full((env.size, env.size), np.nan)
    for s, v in V.items():
        board[s] = v
    im = ax.imshow(board, cmap="viridis")
    for i in range(env.size):
        for j in range(env.size):
            if (i, j) in env.obstacles:
                ax.text(j, i, "X", ha="center", va="center", color="white", fontsize=14)
            elif (i, j) == env.terminal:
                ax.text(j, i, "G", ha="center", va="center", color="white", fontsize=14, fontweight="bold")
            elif (i, j) in policy:
                ax.text(j, i, arrow[policy[(i, j)]], ha="center", va="center", color="white", fontsize=16)
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("Brighter = higher V* (closer to goal / fewer -1 steps)")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------------
# HOW TO READ
# ---------------------------------------------------------------------------
# policy_evaluation: ONE action per state (from π). Sweeps until V stops moving.
# policy_improvement: ALL actions scored with that V; keep the winner.
# policy_iteration:   evaluate → improve → until π does not change.
# value_iteration:    max_a inside the V update; extract π once at the end.
#
# Numbers: V near the goal is less negative (fewer -1 steps left). Walls = X.
# Arrows should point around obstacles toward G. PI and VI should agree.
#
# Next: Monte Carlo — same goal, but learn V/Q from episodes WITHOUT reading P.


In [ ]:
# Monte Carlo Methods
# Generating an Episode

def generate_episode(env, policy):
    episode = []  # list of (state, action, reward)
    state = env.reset()
    done = False
    while not done:
        # For policy, we assume deterministic mapping from state to action
        # If using epsilon-greedy, policy is a function that returns action given state
        if callable(policy):
            action = policy(state)
        else:
            action = policy[state]
        next_state, reward, done, _ = env.step(action)
        episode.append((state, action, reward))
        state = next_state
    return episode